# FindMine detector run 3: wallet, earbuds, medicine fine-tune (Kaggle)

Starts from run 2's weights (which learned keys) and adds Roboflow Universe sets for the weak classes, with the
keys set kept in so keys don't regress. Full steps: `docs/finetune_run3.md`.

1. **Add data** (right panel → *Add Input*):
   - `coco-2017-dataset` (by awsaf49), as in runs 1 and 2
   - your `findmine-labels` dataset (from run 1)
   - **run 2's notebook output**: *Add Input → Notebooks →* your run-2 notebook (it holds `runs/detect/findmine_run2/weights/best.pt`)
   - the keys dataset from run 2 (`keys-computer-vision`) and each new export as its own dataset (Roboflow → Export → **YOLOv8** → zip → upload)
2. **Settings:** Accelerator **GPU T4 x1** (or P100), Internet **On**.
3. **Save Version → Save & Run All (Commit)**. About 2–3 h; you can close the browser.
4. When it's done: version → **Output** → download `out/findmine12_run3.onnx`, `out/findmine12_run3.json`, `out/run3_report.md`
   into `models/` (the .md into `eval/`), then run `ml/compare_detectors.py` on the laptop.


In [ ]:
!pip install -q ultralytics==8.4.182 onnx==1.23.2 onnxslim==0.1.98

In [ ]:
%%writefile build_finetune.py
"""Build a fine-tune dataset: extra YOLO exports (Roboflow Universe sets for weak classes: keys in run 2,
wallet/earbuds/medicine in run 3) on top of a replay sample of the run-1 data, so the detector learns them
without forgetting cups.

    python ml/build_finetune.py --root data/findmine --extra EXPORT_DIR [EXPORT_DIR ...] [--replay 6000]

root   = the run-1 dataset (findmine.yaml, train.txt, val.txt, labels/, images/).
extra  = Roboflow "YOLOv8" exports: data.yaml (names) + {train,valid,test}/{images,labels}/.
Writes into root (Ultralytics finds labels by swapping /images/ for /labels/ in each image path):
    images/extra_<k>/, labels/extra_<k>/   copies of each export, class ids remapped to our 12 classes
    train_ft.txt, val_ft.txt, findmine_ft.yaml
Train: every extra train image + every run-1 image with a weak class (keys, wallet, earbuds, medicine)
+ `replay` random other run-1 images. Val: run-1 val + every extra valid/test image (20% of an export's
train images when it has no valid split).
"""
import argparse
import random
import shutil
from pathlib import Path

CLASSES = ["keys", "wallet", "phone", "glasses", "sunglasses", "remote",
           "earbuds", "medicine", "cup", "bottle", "bag", "watch"]
WEAK = {0, 1, 6, 7}
# Names used by public datasets -> our class. Anything else in an export is dropped.
# Matched after norm(): lower case, "_" and "-" read as spaces.
ALIASES = {"key": "keys", "keys": "keys", "car key": "keys", "house key": "keys",
           "keychain": "keys", "key chain": "keys", "bunch of keys": "keys",
           "wallet": "wallet", "wallets": "wallet", "purse": "wallet", "card holder": "wallet",
           "earbuds": "earbuds", "earbud": "earbuds", "airpods": "earbuds", "airpod": "earbuds",
           "earphones": "earbuds", "earphone": "earbuds", "airpods case": "earbuds", "earbuds case": "earbuds",
           "medicine": "medicine", "medicines": "medicine", "medication": "medicine", "pill bottle": "medicine",
           "pills": "medicine", "pill": "medicine", "medicine box": "medicine", "medicine bottle": "medicine",
           "blister pack": "medicine", "blister": "medicine"}
IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def norm(name):
    return " ".join(name.lower().replace("_", " ").replace("-", " ").split())


def read_names(data_yaml):
    """Class names from a Roboflow data.yaml, without a YAML dependency: `names: [a, b]` or a list/dict block."""
    lines = Path(data_yaml).read_text(encoding="utf-8").splitlines()
    for i, line in enumerate(lines):
        if line.strip().startswith("names:"):
            rest = line.split(":", 1)[1].strip()
            if rest.startswith("["):
                return [n.strip().strip("'\"") for n in rest.strip("[]").split(",") if n.strip()]
            names = []
            for item in lines[i + 1:]:
                s = item.strip()
                if not s or not item.startswith((" ", "\t", "-")):
                    break
                names.append(s.split(":", 1)[1].strip() if ":" in s and not s.startswith("-") else s.lstrip("- "))
            return [n.strip("'\"") for n in names]
    raise SystemExit(f"no names: in {data_yaml}")


def remap(label_text, id_map):
    """YOLO label text -> lines with our class ids; boxes of unmapped classes are dropped."""
    out = []
    for line in label_text.splitlines():
        parts = line.split()
        if len(parts) == 5 and int(parts[0]) in id_map:   # polygons (more than 5 values) are skipped
            out.append(" ".join([str(id_map[int(parts[0])])] + parts[1:]))
    return out


def import_export(export, root, k, seed=0):
    """Copy one export into root as extra_<k>. -> (train paths, val paths) relative to root."""
    export = Path(export)
    names = read_names(export / "data.yaml")
    id_map = {i: CLASSES.index(ALIASES[norm(n)]) for i, n in enumerate(names) if norm(n) in ALIASES}
    if not id_map:
        raise SystemExit(f"{export}: none of its classes {names} map to ours; add them to ALIASES")
    print(f"{export.name}: " + ", ".join(f"{n} -> {CLASSES[id_map[i]]}" for i, n in id_map.items())
          + f" | dropped: {[n for i, n in enumerate(names) if i not in id_map] or 'none'}")
    splits = {"train": [], "val": []}
    for split, dest in (("train", "train"), ("valid", "val"), ("val", "val"), ("test", "val")):
        for img in sorted((export / split / "images").glob("*")) if (export / split / "images").is_dir() else []:
            if img.suffix.lower() not in IMG:
                continue
            lab = export / split / "labels" / (img.stem + ".txt")
            lines = remap(lab.read_text(encoding="utf-8"), id_map) if lab.exists() else []
            if not lines:
                continue                                  # no box of our classes: skip (not a clean negative)
            name = f"{split}_{img.name}"
            (root / "images" / f"extra_{k}").mkdir(parents=True, exist_ok=True)
            (root / "labels" / f"extra_{k}").mkdir(parents=True, exist_ok=True)
            shutil.copy(img, root / "images" / f"extra_{k}" / name)
            (root / "labels" / f"extra_{k}" / (Path(name).stem + ".txt")).write_text("\n".join(lines) + "\n")
            splits[dest].append(f"./images/extra_{k}/{name}")
    if not splits["val"] and len(splits["train"]) >= 5:   # no valid split: hold out 20% of train
        rnd = random.Random(seed)
        rnd.shuffle(splits["train"])
        n = max(1, len(splits["train"]) // 5)
        splits["val"], splits["train"] = splits["train"][:n], splits["train"][n:]
    print(f"  {len(splits['train'])} train, {len(splits['val'])} val images with our classes")
    return splits["train"], splits["val"]


def has_weak(root, img_path):
    lab = root / img_path.replace("/images/", "/labels/", 1).rsplit(".", 1)[0]
    lab = Path(str(lab) + ".txt")
    return lab.exists() and any(line.split()[0].isdigit() and int(line.split()[0]) in WEAK
                                for line in lab.read_text().splitlines() if line.strip())


def build(root, extras, replay=6000, seed=0):
    root = Path(root)
    base_train = (root / "train.txt").read_text().split()
    base_val = (root / "val.txt").read_text().split()
    train, val = [], list(base_val)
    for k, export in enumerate(extras):
        t, v = import_export(export, root, k, seed)
        train += t
        val += v
    weak = [p for p in base_train if has_weak(root, p)]
    weak_set = set(weak)
    rest = [p for p in base_train if p not in weak_set]
    random.Random(seed).shuffle(rest)
    train += weak + rest[:replay]
    (root / "train_ft.txt").write_text("\n".join(train) + "\n")
    (root / "val_ft.txt").write_text("\n".join(val) + "\n")
    names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASSES))
    (root / "findmine_ft.yaml").write_text(f"train: train_ft.txt\nval: val_ft.txt\nnames:\n{names}\n")
    print(f"train {len(train)} images (extra {len(train) - len(weak) - min(replay, len(rest))}, "
          f"run-1 weak-class {len(weak)}, replay {min(replay, len(rest))}); val {len(val)}")
    return train, val


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="data/findmine")
    ap.add_argument("--extra", nargs="+", required=True)
    ap.add_argument("--replay", type=int, default=6000)
    a = ap.parse_args()
    build(a.root, a.extra, a.replay)


In [ ]:
%%writefile train_detector.py
"""Detector training run (GPU box, never AWS). Plan: COCO weights, 12 classes, imgsz 640, 60-80 epochs.

On the GPU machine, from the repo root:
    pip install -r ml/requirements.txt                    # plus a CUDA torch build
    # labels: run ml/merge_dataset.py there, or copy data/findmine/ from a laptop
    mkdir -p data/findmine/images && cd data/findmine/images
    wget http://images.cocodataset.org/zips/train2017.zip http://images.cocodataset.org/zips/val2017.zip
    unzip -q train2017.zip && unzip -q val2017.zip && cd -
    python ml/train_detector.py --epochs 70 --batch -1     # -1 = auto batch for the GPU
Then export for cv.dnn (same settings as ml/spike_yolo_dnn.py):
    yolo export model=runs/detect/findmine_run1/weights/best.pt format=onnx opset=12 imgsz=640 simplify=True
"""
import argparse

from ultralytics import YOLO

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data/findmine/findmine.yaml")
ap.add_argument("--model", default="yolo11n.pt")
ap.add_argument("--epochs", type=int, default=70)
ap.add_argument("--batch", type=int, default=-1)
ap.add_argument("--imgsz", type=int, default=640)
ap.add_argument("--name", default="findmine_run1")
# Fine-tune (run 2, from run 1's best.pt): small lr0, no warm-up (run 1's warm-up hurt epochs 2-4).
ap.add_argument("--lr0", type=float, default=0.01)
ap.add_argument("--warmup-epochs", type=float, default=3.0)
ap.add_argument("--close-mosaic", type=int, default=10)
a = ap.parse_args()

YOLO(a.model).train(
    data=a.data, epochs=a.epochs, batch=a.batch, imgsz=a.imgsz, name=a.name,
    lr0=a.lr0, warmup_epochs=a.warmup_epochs, close_mosaic=a.close_mosaic,
    # Our users: dark rooms, shaky phones, objects cut at the frame edge.
    hsv_v=0.6, degrees=10, translate=0.2, scale=0.6, mosaic=1.0,
    # ponytail: motion blur / low-light gamma+noise come from Ultralytics' built-in albumentations
    # (Blur, MedianBlur, CLAHE) when albumentations is installed; add a custom transform if the
    # FindMine-Home dark/blur split stays weak after run 1.
)


In [ ]:
import glob, os, shutil, subprocess
from pathlib import Path


def find(pattern, depth=6):
    """Matches up to `depth` folders below /kaggle/input. (A recursive ** glob walks all 118k COCO images: minutes.)"""
    return [p for d in range(depth + 1) for p in glob.glob("/kaggle/input/" + "*/" * d + pattern)]


def run(args):
    """Run a script and print its output live: a child process's output never reaches Kaggle's log otherwise."""
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
    if p.returncode:
        raise SystemExit(f"{args[1]} failed (exit {p.returncode}): see the lines above")



print("Inputs attached:")
for p in sorted(glob.glob("/kaggle/input/*/*/*")):
    print("  ", p)
W = Path("/kaggle/working/findmine")
# the run-1 labels dataset; run 2's output also has a findmine/ copy (it has findmine_ft.yaml): skip that one
labels = Path(next(p for p in find("findmine.yaml") if not os.path.exists(Path(p).parent / "findmine_ft.yaml"))).parent
coco = Path(find("train2017/000000000009.jpg")[0]).parent.parent
base = find("runs/detect/findmine_run2/weights/best.pt", 4)
extras = sorted({str(Path(p).parent) for p in find("data.yaml") if not p.startswith(str(labels))})
print("labels:", labels, "| coco:", coco, "| run-2 weights:", base or "NOT FOUND", "| exports:", extras)
assert base, "run 2's best.pt not found: Add Input → Notebooks → your run-2 notebook"
assert extras, ("no export found (a folder with data.yaml + train/images + train/labels). "
                f"Add the datasets under Add Input. Zips seen (Kaggle didn't unpack them?): {find('*.zip')}")

# writable copy of the labels (Ultralytics writes .cache files); never copy image folders
shutil.copytree(labels, W, dirs_exist_ok=True, ignore=shutil.ignore_patterns("images", "*.cache"))
(W / "images").mkdir(exist_ok=True)
for split in ("train2017", "val2017"):
    if not (W / "images" / split).exists():
        os.symlink(coco / split, W / "images" / split)
run(["python", "build_finetune.py", "--root", str(W), "--extra", *extras, "--replay", "6000"])
for lst in ("train_ft.txt", "val_ft.txt"):
    paths = (W / lst).read_text().split()
    missing = [p for p in paths if not (W / p).exists()]
    print(lst, len(paths), "images,", len(missing), "missing")
    assert not missing, missing[:5]


In [ ]:
# From run 2's weights: 30 epochs, small learning rate, no warm-up (same recipe as run 2).
%cd /kaggle/working
run(["python", "train_detector.py", "--data", f"{W}/findmine_ft.yaml", "--name", "findmine_run3",
     "--model", base[0], "--epochs", "30", "--lr0", "0.002", "--warmup-epochs", "0", "--close-mosaic", "5"])


In [ ]:
import json
from ultralytics import YOLO
run = Path("/kaggle/working/runs/detect/findmine_run3")
best = YOLO(run / "weights/best.pt")
out = Path("/kaggle/working/out"); out.mkdir(exist_ok=True)
onnx = Path(best.export(format="onnx", opset=12, imgsz=640, simplify=True, dynamic=False))   # same as run 1 (cv.dnn)
shutil.copy(onnx, out / "findmine12_run3.onnx")
(out / "findmine12_run3.json").write_text(json.dumps([best.names[i] for i in range(len(best.names))]))
shutil.copy(run / "results.csv", out / "results_run3.csv")

# Per-class AP on the fine-tune val set (run-1 val + held-out images of each export), run 2 vs run 3.
rows = ["| class | AP50 run 2 | AP50 run 3 |", "|---|---|---|"]
m2 = best.val(data=str(W / "findmine_ft.yaml"), imgsz=640, split="val", plots=False).box
m1 = YOLO(base[0]).val(data=str(W / "findmine_ft.yaml"), imgsz=640, split="val", plots=False).box
ap = lambda m, i: f"{m.ap50[list(m.ap_class_index).index(i)]:.3f}" if i in m.ap_class_index else "-"
for i, name in best.names.items():
    rows.append(f"| {name} | {ap(m1, i)} | {ap(m2, i)} |")
rows.append(f"| **all (mAP50)** | {m1.map50:.3f} | {m2.map50:.3f} |")
(out / "run3_report.md").write_text("# Detector run 3 (wallet, earbuds, medicine fine-tune)\n\n" + "\n".join(rows) + "\n")
print("\n".join(rows))
